# Imports

Import python libraries as well as the self written FERMI library.

In [ ]:
import sys, os
from os.path import join, split
from getpass import getuser
from glob import glob
from time import strftime
from tqdm.auto import tqdm
from importlib import reload

# Data
import numpy as np
import xarray as xr
import pandas as pd
import h5py

# Images
import imageio
from imageio import imread

# Plotting
import matplotlib.pyplot as plt
from matplotlib.image import NonUniformImage
import matplotlib.gridspec as gridspec
from matplotlib.path import Path

# Scipy
import scipy
from scipy.ndimage import gaussian_filter

# pyFAI
import pyFAI
from pyFAI.azimuthalIntegrator import AzimuthalIntegrator
from pyFAI.detectors import Detector

# Self-written libraries
sys.path.append(os.path.abspath(join(os.pardir,"process_FERMI")))
import helper_functions as helper
import mask_lib
import process_FERMI as pf
import interactive
from interactive import cimshow

plt.rcParams["figure.constrained_layout.use"] = True  # replaces plt.tight_layout

In [ ]:
# interactive plotting
import ipywidgets

%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

# Auto formatting of cells
#load_ext jupyter_black

## Functions

In [ ]:
def preprocess_exp(datafolder, extension, keys=None, sort=False, full_rate=False):
    """
    Loads and preprocesses experiment data from a specified datafolder and file extensions (Dark, Only-Laser, Only-FEL, etc).
    
    Parameters:
        datafolder (str): Path to the folder containing the data.
        extension (str): File extension of the data files. (Dark, Only-Laser, Only-FEL, etc. ...)
        keys (list, optional): Specific HDF5-keys to additionaly extract from the data. Defaults to None.
        sort (bool, optional): Whether to sort the data based on a scan axis. Defaults to False.
        full_rate (bool, optional): If True, filters out empty frames before averaging images. Defaults to False.
    
    Returns:
        pandas.DataFrame: Preprocessed experiment data with computed statistics and loaded images.
    """

    
    # Loading experiment data
    print("Loading: %s" % (datafolder + extension))
    exp = pf.get_exp_dataframe(datafolder + extension, keys=keys)
    for k in ["xgm_UH", "xgm_SH", "diode_sum"]:
        exp[k + "_sum"] = exp[k].apply(np.sum)

    exp["diode_sum_mean"] = exp.diode_sum.apply(np.mean)
    exp["diode_sum_sum"] = exp.diode_sum.apply(np.sum)
    exp["diode_sum_std"] = exp.diode_sum.apply(np.std)
    exp["IR_mean"] = exp.IR.apply(np.mean)
    exp["IR_std"] = exp.IR.apply(np.std)
    #exp["magnet_mean"] = exp.magnet.apply(np.mean)
    #exp["magnet_mean"] = exp.magnet_mean.apply(np.round, args=(3,))
    exp["bunchid"] = exp.bunches.apply(lambda l: l[-1])

    if scan_axis == "index":
        exp["index"] = np.arange(0,len(exp["time"]))
        
    if sort is True:
        exp = exp.sort_values(scan_axis)

    load_images = []
    for idx in range(len(exp["filename"])):
          
            tmp = pf.loadh5(
                    exp["filename"][idx], extra_keys=[ "DPI/AlignZm","PAM/FQPDSum"] 
                )[0].astype("float32")
            tmp[np.isinf(tmp)] = -1
            load_images.append(tmp
                
            )
            print("Loaded %s" % exp["filename"][idx])

    exp["images"] = load_images
    
    return exp

In [ ]:
def filter_false_images(images,filter_thres):
    """
    Identifies and filters out inconsistent images based on intensity deviations.
    
    This function computes the mean intensity of each image, compares it to the 
    ensemble median, and filters out images that deviate beyond a threshold 
    determined by the standard deviation.
    
    Parameters:
        images (numpy.ndarray): Array of images to be analyzed.
        filter_thres (float): Threshold for filtering, defining the acceptable 
                              deviation from the median intensity.
    
    Returns:
        numpy.ndarray: Boolean array indicating valid (True) and filtered (False) images.
    """

    # Calc Monitoring parameter
    image_mean = np.nanmean(images,axis = (-2,-1))
    ensemble_mean = np.nanmedian(image_mean)
    image_std = np.nanstd(image_mean)

    # Filter
    valid = np.abs(image_mean-ensemble_mean) < filter_thres * image_std

    # Plot filter condition
    fig, ax = plt.subplots()
    ax.plot(image_mean,'o-')
    ax.grid()
    ax.set_xlabel("Image Index")
    ax.set_ylabel("Image Mean")
    ax.set_title("Check for inconsistencies of the averaged intensity")
    ax.axhline(ensemble_mean,0,images.shape[0],color = 'g',linestyle = '--')
    ax.axhline((ensemble_mean + filter_thres*image_std),0,images.shape[0],color = 'r',linestyle = '--')
    ax.axhline((ensemble_mean - filter_thres*image_std),0,images.shape[0],color = 'r',linestyle = '--')
    
    return valid

In [ ]:
def dyn_factor(image,image_ref,method = 'scalarproduct', crop=0 ,plot = False, verbose = False):
    '''
    Calculates intensity normalization factor between images
    
    
    Parameters
    ----------
    image: array
        first image
        
    image_ref: array
        reference image
    
    method: str
        Method for calculating scaling factor (scalarproduct,correlation)
    
    crop : int
        crop array from each side for calc of factor and offset
    
    plot : bool
        Plot fit if method is correlation
        
    verbose : bool
        print factor and offset
    
    Returns
    -------
    factor: scalar
        Intensity correction factor
    -------
    author: CK 2023
    '''

    #Fit (lin)
    def func(x, a, b):
        return a*x+b
    
    #Do you crop the images?
    if crop == 0:
        crop_s = slice(None)
    elif crop > 0:
        crop_s = np.s_[crop:-crop,crop:-crop]
    
    if method == 'scalarproduct':
        factor = np.sum(image[crop_s]*image_ref[crop_s])/np.sum(image_ref[crop_s]*image_ref[crop_s])
        offset = 0
        
        if verbose == True:
            print(f'Intensity correction factor:', factor)
        
    elif method == 'correlation':
        #Create y, x data
        xdata = np.concatenate(image_ref[crop_s])
        ydata = np.concatenate(image[crop_s])
        
        #Ignore all x,y = 0 values, e.g., if a mask is used
        ignore = np.logical_or((xdata==0),(ydata==0))
        xdata = xdata[np.argwhere(ignore == False)]
        ydata = ydata[np.argwhere(ignore == False)]
        
        xdata = np.squeeze(xdata,axis=1)
        ydata = np.squeeze(ydata,axis=1)
        
        #Fitting
        popt, pcov = scipy.optimize.curve_fit(func, xdata, ydata)
        factor = popt[0]
        offset = popt[1]
        
        if verbose == True:
            print(f'Linear Fit: {factor:0.4f}*x + {offset:0.4f}')
        
        if plot == True:
            fig, ax = plt.subplots()
            ax.plot()
            ax.scatter(xdata, ydata, s= 5)
            ax.plot(xdata,func(xdata,*popt),'r-')
            ax.set_xlabel('Intensity Ref')
            ax.set_ylabel('Intensity')
            ax.set_title(f'Linear Fit: {factor:0.4f}*x + {offset:0.4f}')
            plt.tight_layout()
            
    return factor, offset

# Experimental details

In [ ]:
# Define basic folders
BASEFOLDER = r"/data/beamtimes/FERMI/2503_chiral"
PROPOSAL = "20234051"
USER = getuser()

In [ ]:
# Dict with most basic experimental parameter
experimental_setup = {
    "px_size": 13.5e-6,  # pixel_size of camera
    "binning": 2,  # Camera binning
}

# Setup for azimuthal integrator
detector = Detector(
    experimental_setup["binning"] * experimental_setup["px_size"],
    experimental_setup["binning"] * experimental_setup["px_size"],
)

# General saving folder
folder_target = pf.create_folder(join("/data/beamtimes/FERMI/2503_chiral/results", "Log"))
print("Output Folder: %s" % folder_target)

# Load Data

## Define Scans

In [ ]:
# Define for loading
sample = "SP200622_R4"
scan_name = "Chiral_CL_Sky_Scan" 

# Is it a pumped scan?
pump_mode = "OF_BG" # None, "hysteresis", "IR_fluence", ["OL_OF_BG"], ["OF_BG"]

if pump_mode == "IR_fluence":
    scan_axis = "IR_mean"
    extension = "_OF"
    
elif pump_mode == "hysteresis":
    scan_axis = "magnet"
    extension = "_OF"
    
elif pump_mode is None:
    scan_axis ="index"# "IR_mean"#"magnet"#,"magnet", "index"
    extension = "_OF"

# Which keys to load in addition to default
extra_keys = {
    "diode_sum": "PAM/FQPDSum",
    "IR": "Laser/Energy2",
    "magnet": "DPI/CoilCurrent",
    #"magnet_waveform": "DPI/CoilWaveform",
    "bunches": "bunches",
    "time": "",
    "samplex": "DPI/SampleX",
    "sampley": "DPI/SampleY",
    "camera_rot": "/DPI/CCDTheta",
    "sample_rot": "/DPI/SampleTheta"
}

# Create savefolder
fsave = pf.create_folder(join("/data/beamtimes/FERMI/2503_chiral/results/Log", sample, scan_name)) #"/data/export/cklose/2503_FERMI_Chiral_Scattering/Analysis"
print("Save Folder: %s"%fsave)

## Load images

### Load Scan images

In [ ]:
# Folder for loading
extension = "_OF"
scan_name_pos = "Chiral_CR_Sky_Scan" 
scan_id_pos = 43

# Loading experiment data pos
scan_pos = f"%s_%03d" % (scan_name_pos, scan_id_pos)
samplefolder = join(sample, scan_pos)
datafolder = join(BASEFOLDER, samplefolder)
exp_pos = preprocess_exp(datafolder, extension, keys=extra_keys)


extension = "_OF"
scan_name_neg = "Chiral_CL_Sky_Scan" 
scan_id_neg = 43

# Loading experiment data neg
scan_neg = f"%s_%03d" % (scan_name_neg, scan_id_neg)
samplefolder = join(sample, scan_neg)
datafolder = join(BASEFOLDER, samplefolder)
exp_neg = preprocess_exp(datafolder, extension, keys=extra_keys)

# Add wavelength and distance
experimental_setup["lambda"] = exp_pos["wavelength"][0] * 1e-9
experimental_setup["ccd_dist"] = (exp_pos["ccdz"][0] + 50) * 1e-3
experimental_setup["camera_rot"] = exp_pos["camera_rot"][0]
experimental_setup["sample_rot"] = exp_pos["sample_rot"][0]

print("Data loaded!")

In [ ]:
# What did you scan?
fig, ax = plt.subplots()
fig.suptitle("ScanId: %03d"%scan_id_pos)
ax.plot(np.arange(len(exp_pos)), exp_pos[scan_axis], "-o")
ax.set_xlabel("Index")
ax.set_ylabel(scan_axis)
ax.grid()

fig, ax = plt.subplots()
fig.suptitle("ScanId: %03d"%scan_id_neg)
ax.plot(np.arange(len(exp_neg)), exp_neg[scan_axis], "-o")
ax.set_xlabel("Index")
ax.set_ylabel(scan_axis)
ax.grid()

In [ ]:
fig, ax = cimshow(np.stack(exp_pos.images))
ax.set_title("Image Slideshow viewer: ScanId: %03d"%scan_id_pos)
plt.show()

fig, ax = cimshow(np.stack(exp_neg.images))
ax.set_title("Image Slideshow viewer: ScanId: %03d"%scan_id_neg)

In [ ]:
pos_mean = np.mean(np.stack(exp_pos.images),axis=0)
neg_mean = np.mean(np.stack(exp_neg.images),axis=0)
tmp = (pos_mean-neg_mean)#/(pos_mean+neg_mean)
tmp = gaussian_filter(tmp,2)
cimshow(tmp,cmap="coolwarm") #*(1-mask)

### Dark images

In [ ]:
# Loading experiment data
extension = "_BG"

# Load data
datafolder_BG = join(BASEFOLDER, sample,f"%s_%03d" % (scan_name, scan_id_pos))
datafolder_BG = "/data/beamtimes/FERMI/2503_chiral/SP200622_R4/Chiral_CL_Sky_Scan_042"
if os.path.exists(datafolder_BG + extension):
    exp_bg_pos = preprocess_exp(datafolder_BG, extension, keys=extra_keys)
    exp_bg_pos = exp_bg_pos.sort_values("time")

    if False: #pumped_images
        dark_pos = np.stack(exp_bg_pos["images"])
    else:
        dark_pos = np.mean(np.stack(exp_bg_pos["images"]), axis=0)

    print("Data loaded!")
else:
    print("No Folder: %s"%datafolder_BG + extension)
    dark_pos = np.zeros(exp_pos["images"][0].shape)

# Load data
datafolder_BG = join(BASEFOLDER, sample,f"%s_%03d" % (scan_name, scan_id_neg))
datafolder_BG = "/data/beamtimes/FERMI/2503_chiral/SP200622_R4/Chiral_CL_Sky_Scan_042"
if os.path.exists(datafolder_BG + extension):
    exp_bg_neg = preprocess_exp(datafolder_BG, extension, keys=extra_keys)
    exp_bg_neg = exp_bg_neg.sort_values("time")

    if False: #pumped_images
        dark_neg = np.stack(exp_bg_neg["images"])
    else:
        dark_neg = np.mean(np.stack(exp_bg_neg["images"]), axis=0)

    print("Data loaded!")
else:
    print("No Folder: %s"%datafolder_BG + extension)
    dark_neg = np.zeros(exp_neg["images"][0].shape)

# Plot images
fig, ax = cimshow(dark_pos)
fig.set_size_inches(6, 6)
ax.set_title("Dark Image: %03d"%scan_id_pos)
plt.show()

# Plot images
fig, ax = cimshow(dark_neg)
fig.set_size_inches(6, 6)
ax.set_title("Dark Image: %03d"%scan_id_neg)

### Laser only

In [ ]:
if pump_mode == "IR_fluence":
    extension = "_OL"
    
    # Loading experiment data pos
    scan = f"%s_%02d" % (scan_name_pos, scan_id_pos)
    samplefolder = join(sample, scan)
    datafolder = join(BASEFOLDER, samplefolder)
    
    # Loading experiment data
    exp_ol_pos = preprocess_exp(datafolder, extension, keys=extra_keys)
    exp_ol_pos = exp_ol_pos.sort_values("time")

    dark_ol_pos = np.stack(exp_ol_pos["images"])

    # Plot images
    fig, ax = cimshow(dark_ol_pos)
    fig.set_size_inches(6, 6)
    ax.set_title("Only Laser Images Pos")

    
    # Loading experiment data neg
    scan = f"%s_%02d" % (scan_name_pos, scan_id_pos)
    samplefolder = join(sample, scan)
    datafolder = join(BASEFOLDER, samplefolder)
    
    # Loading experiment data
    exp_ol_neg = preprocess_exp(datafolder, extension, keys=extra_keys)
    exp_ol_neg = exp_ol_pos.sort_values("time")

    dark_ol_neg = np.stack(exp_ol_neg["images"])

    # Plot images
    fig, ax = cimshow(dark_ol_neg)
    fig.set_size_inches(6, 6)
    ax.set_title("Only Laser Images Pos")

    print("Data loaded!")
else:
    print("Not executed for static hysteresis!")

### FEL only

In [ ]:
if (pump_mode == "hysteresis") or (pump_mode == "IR_fluence"):
    extension = "_OF"
    
    # Loading experiment data pos
    #scan = f"%s_%02d" % (scan_name_pos, scan_id_pos)
    scan = f"Chiral_CL_Saturated_Scan_%0.2d"%scan_id_pos 
    samplefolder = join(sample, scan)
    datafolder = join(BASEFOLDER, samplefolder)
    
    # Loading experiment data
    exp_of_pos = preprocess_exp(datafolder, extension, keys=extra_keys)
    exp_of_pos = exp_of_pos.sort_values("time")

    dark_of_pos = np.stack(exp_of_pos["images"])

    # Plot images
    fig, ax = cimshow(dark_of_pos)
    fig.set_size_inches(6, 6)
    ax.set_title("Only FEL Images Pos")

    
    # Loading experiment data neg
    #scan = f"%s_%02d" % (scan_name_pos, scan_id_pos)
    scan = f"Chiral_CR_Saturated_Scan_%0.2d"%scan_id_pos 
    samplefolder = join(sample, scan)
    datafolder = join(BASEFOLDER, samplefolder)
    
    # Loading experiment data
    exp_of_neg = preprocess_exp(datafolder, extension, keys=extra_keys)
    exp_of_neg = exp_of_pos.sort_values("time")

    dark_of_neg = np.stack(exp_of_neg["images"])

    # Plot images
    fig, ax = cimshow(dark_of_neg)
    fig.set_size_inches(6, 6)
    ax.set_title("Only FEL Images Pos")

    print("Data loaded!")
else:
    print("Not executed for static measurement!")

### Subtract dark images (for dynamic hysteresis also OL, OF) and normalize images to I0

In [ ]:
# Incident intensity
fig, ax = plt.subplots()
ax.plot(np.arange(len(exp_neg)), exp_neg["diode_sum_sum"])
ax.set_xlabel("Image Index")
ax.set_ylabel("Incident Intensity (a.u.)")

In [ ]:
# Which key to use for normalization?
norm_key = "diode_sum_sum"
filter_key = "diode_sum_sum"

# For positive helicity
if (pump_mode == "hysteresis") or (pump_mode == "IR_fluence"):
    # Loop over images
    pos = []
    pos_of = []
    for index, r in tqdm(exp_pos.iterrows(), total=len(exp_pos)):
        if (pump_mode == "hysteresis"):
            # Find closest dark image in time series
            idx = np.argmin(abs(r.time - exp_bg_pos.time))
            im_bg = exp_bg_pos.iloc[idx]["images"]
    
            # Find closest only laser image in time series
            idx = np.argmin(abs(r.time - exp_ol_pos.time))
            im_ol = exp_ol_pos.iloc[idx]["images"]
    
            # Find closest only fel image in time series
            idx = np.argmin(abs(r.time - exp_of_pos.time))
            im_of = exp_of_pos.iloc[idx]["images"]
    
            # Subtract background
            im = (r.images - im_ol) / r[norm_key]
            of_norm = (im_of - im_bg) / exp_of_pos.iloc[idx][norm_key]
            im = im - of_norm


        if pump_mode == "IR_fluence":
            im_bg = dark_pos.copy()

            """
            # Find closest dark image in time series
            idx = np.argmin(abs(r.time - exp_bg_pos.time))
            im_bg = exp_bg_pos.iloc[idx]["images"]
            """
            # Find closest only fel image in time series
            idx = np.argmin(abs(r.time - exp_of_pos.time))
            im_of = exp_of_pos.iloc[idx]["images"]
            
            # Subtract background
            im = (r.images - im_bg) / r[norm_key]
            of_norm = (im_of - im_bg) / exp_of_pos.iloc[idx][norm_key]
            im = im - of_norm

        pos.append(im)
        pos_of.append(of_norm)

else:
    pos = np.stack(exp_pos.images) - dark_pos
    pos = pos / np.broadcast_to(np.array(exp_pos["diode_sum_mean"]),np.stack(exp_pos.images).T.shape).T
pos = np.stack(pos)
pos_mean = np.mean(pos, axis=0)

# For negative helicity
if (pump_mode == "hysteresis") or (pump_mode == "IR_fluence"):
    # Loop over images
    neg = []
    neg_of = []
    for index, r in tqdm(exp_neg.iterrows(), total=len(exp_neg)):
        if (pump_mode == "hysteresis"):
            # Find closest dark image in time series
            idx = np.argmin(abs(r.time - exp_bg_neg.time))
            im_bg = exp_bg_neg.iloc[idx]["images"]
    
            # Find closest only laser image in time series
            idx = np.argmin(abs(r.time - exp_ol_neg.time))
            im_ol = exp_ol_neg.iloc[idx]["images"]
    
            # Find closest only fel image in time series
            idx = np.argmin(abs(r.time - exp_of_neg.time))
            im_of = exp_of_neg.iloc[idx]["images"]
    
            # Subtract background
            im = (r.images - im_ol) / r[norm_key]
            of_norm = (im_of - im_bg) / exp_of_neg.iloc[idx][norm_key]
            im = im - of_norm


        if pump_mode == "IR_fluence":
            im_bg = dark_neg.copy()
            """
            # Find closest dark image in time series
            idx = np.argmin(abs(r.time - exp_bg_neg.time))
            im_bg = exp_bg_neg.iloc[idx]["images"]
        	"""
            # Find closest only fel image in time series
            idx = np.argmin(abs(r.time - exp_of_neg.time))
            im_of = exp_of_neg.iloc[idx]["images"]
    
            # Subtract background
            im = (r.images - im_bg) / r[norm_key]
            of_norm = (im_of - im_bg) / exp_of_neg.iloc[idx][norm_key]
            im = im - of_norm

        neg.append(im)
        neg_of.append(of_norm)

else:
    neg = np.stack(exp_neg.images) - dark_neg
    neg = neg / np.broadcast_to(np.array(exp_neg["diode_sum_mean"]),np.stack(exp_neg.images).T.shape).T
neg = np.stack(neg)
neg_mean = np.mean(neg, axis=0)

# Calc diff
images = neg-pos
im_mean = np.mean(neg, axis=0)-np.mean(pos, axis=0)

In [ ]:
IR = np.stack(exp_neg["IR_mean"])

Data = (pos-neg)#*(1-mask)
vmin, vmax = np.nanpercentile(Data[Data!=0], [1, 99])
fig, ax = plt.subplots(figsize=(8,6))

for idx, dat in enumerate(Data):
    fig.suptitle(f"IR-Energy {IR[idx]:.1f}µ, {scan_pos} - {scan_neg} ")
    ax.imshow(dat, vmin=vmin, vmax=vmax,cmap="coolwarm")

    path = join(fsave, "Difference_%03d_%03d_IR%.0f_Sat1A_%s.png" % (scan_id_pos, scan_id_neg, IR[idx], USER))
    
    #fig.savefig(path)


## Draw beamstop mask

In [ ]:
poly_mask = interactive.draw_polygon_mask(pos_mean)

In [ ]:
# Take poly coordinates and mask from widget
p_coord = poly_mask.get_vertice_coordinates()
mask = poly_mask.full_mask.astype(int)

cimshow(mask)

print("Mask coordinates: %s" % p_coord)

In [ ]:
def load_poly_coordinates():
    """
    Dictionary that stores polygon corner coordinates of all drawn masks
    Example: How to add masks with name "test":
    mask_coordinates["test"] = copy coordinates from above
    """

    # Setup dictonary
    mask_coordinates = dict()

    mask_coordinates["bs_streak"] = [[(437.0, 717.8), (441.8, 804.3), (548.1, 790.3), (610.6, 775.3), (625.3, 732.2)]]
    mask_coordinates["bs_pan_scan_>23"] =  [[(589.9, 380.3), (386.6, 380.3), (383.9, 589.0), (598.1, 590.4), (596.7, 558.8), (615.9, 521.7), (647.5, 514.9), (1036.5, 517.8), (1039.3, 449.2), (681.2, 447.8), (640.1, 440.9), (607.1, 430.0)]]
    mask_coordinates["bs_pan_100mm"] = [[(414.8, 407.7), (416.2, 597.1), (609.8, 598.5), (611.2, 562.8), (624.9, 539.5), (648.3, 529.9), (1034.2, 533.9), (1032.8, 466.7), (674.2, 469.5), (650.4, 469.5), (623.8, 458.3), (616.8, 451.3), (607.0, 433.1), (600.0, 400.9)]]
    mask_coordinates["parasitic_scattering"] = [[(476.9, 393.0), (411.0, 136.3), (272.3, 169.2), (351.9, 395.8), (332.7, 446.6), (331.3, 483.7), (331.3, 511.1), (340.9, 527.6), (357.4, 566.1), (369.8, 583.9), (376.6, 634.7), (380.8, 686.9), (426.1, 719.8), (504.3, 719.8), (551.0, 703.4), (566.1, 659.4), (545.5, 552.3)]]
    return mask_coordinates

In [ ]:
# Which drawn masks do you want to load?
polygon_names = ["bs_pan_scan_>23","bs_streak","parasitic_scattering"] 
mask = mask_lib.load_poly_masks(pos[0].shape,load_poly_coordinates(),polygon_names)

fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
mi, ma = np.percentile(pos_mean, [1, 99])
ax[0].imshow(pos_mean * (1 - mask), vmin=mi, vmax=ma)
ax[0].set_title("(1-mask)")
ax[1].imshow(pos_mean * mask, vmin=mi, vmax=ma)
ax[1].set_title("mask")

In [ ]:
# Use widget to shift and expand or shrink the mask
ss_mask = interactive.Shift_Scale_Mask(pos_mean, mask, shift=[0, 0], scale=1)

In [ ]:
# Take mask, shift and scaling from widget
mask, mask_shift, mask_scale = ss_mask.get_mask()

## Find center

### Basic widget to find center

Try to **align** the circles to the **center of the scattering pattern**. Care! Position of beamstop might be misleading and not represent the actual center of the hologram. 

In [ ]:
# Set center position via widget
ic = interactive.InteractiveCenter(pos_mean,c0=476,c1 = 426,rBS=95)

In [ ]:
# Get center positions
center = [ic.c0, ic.c1]
print(f"Center:", center)

### Azimuthal integrator widget for finetuning

In [ ]:
# Setup azimuthal integrator for virtual geometry
rot2 = -np.deg2rad(experimental_setup["camera_rot"]-experimental_setup["sample_rot"]) 
offset1 = -np.tan(rot2)*experimental_setup["ccd_dist"]
poni1 = center[0]* experimental_setup["px_size"]* experimental_setup["binning"] + offset1
poni2 = center[1]* experimental_setup["px_size"]* experimental_setup["binning"]

ai = interactive.AzimuthalIntegrator(
    dist=experimental_setup["ccd_dist"],
    detector=detector,
    wavelength=experimental_setup["lambda"],
    poni1=poni1,  # y (vertical)
    poni2=poni2,  # x (horizontal)
    rot2 = rot2,
)

In [ ]:
# Plotting to find  relevant q range
I_t, q_t, phi_t = ai.integrate2d(
    im_mean,
    200,
    #radial_range=(0.004, 0.009),
    unit="q_nm^-1",
    correctSolidAngle=False,
    dummy=np.nan,
    mask=mask,
    method="bbox"
)
az2d = xr.DataArray(I_t, dims=("phi", "q"), coords={"q": q_t, "phi": phi_t})

# Plot
fig, ax = plt.subplots()
mi, ma = np.nanpercentile(I_t, [1, 99])
az2d.plot.imshow(ax=ax, vmin=mi, vmax=ma)
plt.title(f"Azimuthal integration")

# Vertical lines
# q_lines = [0.025, 0.05]
# for qt in q_lines:
#    ax.axvline(qt, ymin=0, ymax=180, c="red")

# Calculate Difference

In [ ]:
# Get scaling factor and offset
factor, offset = dyn_factor(
    pos_mean * (1 - mask),
    neg_mean * (1 - mask),
    method="correlation",
    verbose=True,
    plot=True,
)
# factor = 1

# Calculate differences (magnetic) and sums (topographc) contrast holograms.
# _c: centered, without beamstop, _b: centered, with beamstop
diff =pos_mean / factor - neg_mean - offset

In [ ]:
# Create figure
tmp = scipy.ndimage.gaussian_filter(diff,2 )*(1-mask)
#tmp = diff*(1-mask)

#Plotting
#fig, ax = cimshow(tmp*(1-mask), cmap="coolwarm")
vmin, vmax = np.nanpercentile(tmp[tmp!=0],[1,99])
fig, ax = plt.subplots(figsize=(8,7))
m= ax.imshow(tmp,vmin=vmin,vmax=vmax,cmap="coolwarm")
ax.set_title(f"IR-Energy {IR[idx]:.1f}µ, {scan_pos} - {scan_neg} ")
plt.colorbar(m)

# Save difference image
fname = join(fsave, "Difference_%03d_%03d_Sat1A_%s.png" % (scan_id_pos, scan_id_neg,  USER))
print("Saving: %s" % fname)
#plt.savefig(fname)

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(12,4),sharex=True,sharey=True)

tmp = pos_mean*(1-mask)
vmin, vmax = np.nanpercentile(tmp[tmp!=0],[1,99])
ax[0].imshow(tmp,vmin=vmin,vmax=vmax)
ax[0].set_title(scan_pos)

tmp = neg_mean*(1-mask)
#vmin, vmax = np.nanpercentile(tmp[tmp!=0],[1,99])
ax[1].imshow(tmp,vmin=vmin,vmax=vmax)
ax[1].set_title(scan_neg)

tmp = diff*(1-mask)
vmin, vmax = np.nanpercentile(tmp[tmp!=0],[1,99])
ax[2].imshow(tmp,vmin=vmin,vmax=vmax)
ax[2].set_title(f"{scan_pos}-{scan_neg}")

# Azimuthal Integration

In [ ]:
# Setup azimuthal integrator for virtual geometry
rot2 = -np.deg2rad(experimental_setup["camera_rot"]-experimental_setup["sample_rot"]) 
offset1 = -np.tan(rot2)*experimental_setup["ccd_dist"]
poni1 = center[0]* experimental_setup["px_size"]* experimental_setup["binning"] + offset1
poni2 = center[1]* experimental_setup["px_size"]* experimental_setup["binning"]

ai = interactive.AzimuthalIntegrator(
    dist=experimental_setup["ccd_dist"],
    detector=detector,
    wavelength=experimental_setup["lambda"],
    poni1=poni1,  # y (vertical)
    poni2=poni2,  # x (horizontal)
    rot2 = rot2,
)

In [ ]:
# Do 2d Azimuthal integration of all images and add to xarray
#list_i2d = []
"""
for im in tqdm(images):
    i2d, q, chi = ai.integrate2d(im, 500, 90, dummy=np.nan, mask=mask, method="bbox")
    list_i2d.append(i2d)
    
# Setup xarray
data = xr.Dataset()
data["images"] = xr.DataArray(images, dims=["index", "y", "x"])
data[scan_axis] = xr.DataArray(exp_pos[scan_axis], dims=["index"])
data["q"] = q
data["chi"] = chi
data["i2d"] = xr.DataArray(list_i2d, dims=["index", "chi", "q"])
data = data.assign_attrs({"center": center})
"""

# Calculate azimuthal integration of pos, neg and diff
i2d_pos, q, chi = ai.integrate2d(pos_mean, 200, 90, dummy=np.nan,mask=mask,method="bbox",)
i2d_neg, q, chi = ai.integrate2d(neg_mean, 200, 90, dummy=np.nan,mask=mask, method="bbox",)
i2d_diff, q, chi = ai.integrate2d(diff, 200, 90, dummy=np.nan,mask=mask, method="bbox",)
i2d_test, q, chi = ai.integrate2d(np.ones(pos_mean.shape), 200, 90, dummy=np.nan,mask=mask, method="bbox",)
#i2d_diff, q, chi = ai.integrate2d(diff, 200, 90, dummy=np.nan, mask=mask, method="bbox",)

# Add to xarray
data = xr.Dataset()
data["q"] = q
data["chi"] = chi

data["pos"] = xr.DataArray(pos_mean, dims=["y", "x"])
data["neg"] = xr.DataArray(neg_mean, dims=["y", "x"])
data["diff"] = xr.DataArray(diff, dims=["y", "x"])

data["i2d_pos"] = xr.DataArray(i2d_pos, dims=["chi", "q"])
data["i2d_neg"] = xr.DataArray(i2d_neg, dims=["chi", "q"])
data["i2d_diff"] = xr.DataArray(i2d_diff, dims=["chi", "q"])
data["i2d_test"] = xr.DataArray(i2d_test, dims=["chi", "q"])
data = data.assign_attrs({"center": center})

# If it's a pumped hysteresis, do it also for the OF case
#if False: #pumped_hysteresis
#    list_i2d_of = []
#    for im in tqdm(images_of):
#        i2d, q, chi = ai.integrate2d(im, 500, 90, dummy=np.nan, mask=mask)
#        list_i2d_of.append(i2d)#
#
#    # Setup xarray
#    data["images_of"] = xr.DataArray(images_of, dims=["index", "y", "x"])
#    data["i2d_of"] = xr.DataArray(list_i2d_of, dims=["index", "chi", "q"])

## Select relevant chi-range

In [ ]:
# Plot 2d and 1d azimuthal integration to estimate the relevant chi and q range
# which image to show?
idx = 0

# Which chi-mode? (all,hetero,homo)
chi_mode = "non_streak"

# Define chi-range
if chi_mode == "all":
    sel_chi = (data.chi <= 180) * (data.chi >= -180)
elif chi_mode == "non_streak":
    sel_chi = (data.chi < 120) * (data.chi > 60)
    sel_chi = ~ sel_chi

# Select chi range
data["i1d_pos"] = data.i2d_pos.where(sel_chi, drop=True).mean("chi")
data["i1d_neg"] = data.i2d_neg.where(sel_chi, drop=True).mean("chi")
data["i1d_diff"] = data.i2d_diff.where(sel_chi, drop=True).mean("chi")
data["i1d_test"] = data.i2d_test.where(sel_chi, drop=True).mean("chi")

# Plot
fig, ax = plt.subplots(
    2,
    1,
    figsize=(8, 8),
    sharex=True,
)
mi, ma = np.nanpercentile(data["i2d_pos"],[0.1, 99.99])
data["i2d_pos"].where(sel_chi).plot.imshow(ax=ax[0], vmin=mi, vmax=ma,cmap="coolwarm")
ax[0].set_title(f"2d Azimuthal integration")
ax[0].grid()

# Plot 1d azimuthal integration to estimate the relevant q-range
ax[1].plot(data.q, data.i1d_pos,label="pos")
ax[1].plot(data.q, data.i1d_neg,label="neg")
ax[1].plot(data.q, data.i1d_diff,label="diff")
#ax[1].plot(data.q, data.i1d_test,label="test")
#ax[1].set_yscale("log")
ax[1].set_title("1d Azimuthal Integration")
ax[1].grid()
ax[1].set_ylabel("Integrated intensity")
ax[1].set_xlabel("q")
ax[1].legend()

## Test normalization

In [ ]:
# Subtract offset at large q
# choose the relevant q-range which shows plateau
q0, q1 = 0.016, 0.018

# remove offset
sel = (data.q > q0) * (data.q < q1)
offset = data["i1d_pos"].where(sel).mean()
data["i2d_pos"] = data["i2d_pos"] - offset
data["i1d_pos"] = data["i1d_pos"] - offset
data["pos"] = data["pos"]-offset/data["i1d_test"].where(sel).mean()

offset = data["i1d_neg"].where(sel).mean()
data["i2d_neg"] = data["i2d_neg"]-offset
data["i1d_neg"] = data["i1d_neg"]-offset
data["neg"] = data["neg"]-offset/data["i1d_test"].where(sel).mean()

## Normalize by peak intensity
# choose the relevant q-range which shows the peak
q0, q1 = 0.005, 0.011

# normalize to 1
sel = (data.q > q0) * (data.q < q1)
factor = data["i1d_pos"].where(sel).max()
data["i1d_pos"] = data["i1d_pos"]/factor
data["pos"] = data["pos"]/factor*data["i1d_test"].where(sel).mean()
factor = data["i1d_neg"].where(sel).max()
data["i1d_neg"] = data["i1d_neg"]/factor
data["neg"] = data["neg"]/factor*data["i1d_test"].where(sel).mean()

# Plot results
fig, ax = plt.subplots(figsize=(8,8))
# Plot 1d azimuthal integration to estimate the relevant q-range
ax.plot(data.q, data.i1d_pos, label="Pos")
ax.plot(data.q, data.i1d_neg, label="Neg")
ax.set_title("1d Azimuthal Integration")
ax.set_ylabel("Integrated intensity")
ax.set_xlabel("q")
ax.grid()
ax.legend()

In [ ]:
cimshow([(pos_mean-neg_mean)*(1-mask)])

In [ ]:
# Create figure
tmp = scipy.ndimage.gaussian_filter(data.pos.values-data.neg.values,1 )*(1-mask)
#tmp = (data.pos.values-data.neg.values)*(1-mask)

#Plotting
#fig, ax = cimshow(tmp*(1-mask), cmap="coolwarm")
vmin, vmax = np.nanpercentile(tmp[tmp!=0],[1,99])
fig, ax = plt.subplots(figsize=(8,7))
m= ax.imshow(tmp-scipy.ndimage.gaussian_filter(np.mean(tmp[-50:,:], axis=0), 9),vmin=vmin,vmax=vmax,cmap="coolwarm") #-(np.mean(tmp[-200:,:], axis=0)+np.mean(tmp[0:100,:], axis=0))/2
ax.set_title(f"IR-Energy {IR[idx]:.1f}µ, {scan_pos} - {scan_neg} ")
plt.colorbar(m)

# Save difference image
fname = join(fsave, "Difference_%03d_%03d_Domain_%s.png" % (scan_id_pos, scan_id_neg,  USER))
print("Saving: %s" % fname)
#plt.savefig(fname)

In [ ]:
cimshow(tmp-scipy.ndimage.gaussian_filter(np.mean(tmp[-50:,:], axis=0), 9), cmap = "jet")

In [ ]:
plt.figure()
plt.plot(scipy.ndimage.gaussian_filter(np.mean(tmp[-50:,:], axis=0), 9))

## Select relevant q-range

In [ ]:
# Select relevant q-range for averaging
q0, q1 = 0.01, 0.04
binning = False
bins = []

# Get SAXS from q-range
sel = (data.q > q0) * (data.q < q1)
data["saxs"] = data.i1d_pos.where(sel, drop=True).mean("q")
if False: #pumped_hysteresis
    data["saxs_of"] = data.i1d_of.where(sel, drop=True).mean("q")

# Averaging of same scan axis values or binning
if binning is True:
    # Execute binning
    data_bin = data.groupby_bins(scan_axis, bins).mean()

    # Rename binned values, drop intervals as those cannot be save in h5
    bin_scan_axis = scan_axis + "_bins"
    data_bin = data_bin.swap_dims({bin_scan_axis: scan_axis})
    data_bin = data_bin.drop(bin_scan_axis)
else:
    _, count = np.unique(data[scan_axis].values, return_counts=True)
    if np.any(count > 1):
        data_bin = data.groupby(scan_axis).mean()
    else:
        data_bin = data.swap_dims({"index": scan_axis})

# To create log plot
data_bin["i1dlog"] = np.log10(data_bin["i1d"]+ 1)

# Add scan identifier
data_bin["scan"] = scan

# Add AI mask
data_bin["mask"] = xr.DataArray(mask, dims=["y", "x"])

# Direction of "time"
if np.sum(data[scan_axis][1:].values - data[scan_axis][0:-1].values) >= 0:
    data_bin["order"] = 1
elif np.sum(data[scan_axis][1:].values - data[scan_axis][0:-1].values) < 0:
    data_bin["order"] = -1

# Plot
if False: #pumped_hysteresis
    label_set = "pump effect (IM-OF-OL)"
else:
    label_set = "only FEL (static)"
fig, ax = plt.subplots()
ax.plot(
    data_bin[scan_axis].values,
    data_bin["saxs"].values,
    "o-",
    label=label_set,
)
if False: #pumped_hysteresis
    ax.plot(
        data_bin[scan_axis].values,
        data_bin["saxs_of"].values,
        "o-",
        label="only FEL (OF)",
    )
# ax.plot(
#    data_bin[scan_axis].values,
#    np.mean(data_bin["images"].values * (1 - mask), axis=(1, 2)),
#    label="Simple Mean",
# )

ax.set_xlabel(scan_axis)
ax.set_ylabel("Integrated SAXS")
ax.set_title("Scan: %s (Azimuthal Integration)" % scan)
ax.grid()
ax.legend()

# Save fig
fname = join(fsave, "Hysteresis_%s_%s_%s.png" % (scan, chi_mode, USER))
print("Saving: %s" % fname)
plt.savefig(fname)

# Hysteresis Plot

## Select roi for plotting

How to use:
1. Zoom into the image and adjust your FOV until you are satisfied.
2. Save the axes coordinates.

In [ ]:
fig, ax = cimshow(data_bin["images"].values)

In [ ]:
# Takes start and end of x and y axis
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi = np.array([int(y1), int(y2), int(x1), int(x2)])
roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
print(f"Roi:", roi)

## Plotting

In [ ]:
# Find max and min considering all images
allmin, allmax = np.nanpercentile(data_bin["i2d"].values, [1, 99.9])
# allmin, allmax = np.nanpercentile(images - images[0], [3, 97])

if allmin < .1:
    allmin = .1

print("Min: %d Max: %d" % (allmin, allmax))

# Create folder for gif single frames
folder_gif = helper.create_folder(join(fsave, "Scan_%s" % scan))

im_fnames = []
for i in tqdm(range(len(data_bin[scan_axis].values))):
    # Plot for averaged image
    fig = plt.figure(figsize=(6, 10))
    gs1 = gridspec.GridSpec(
        4,
        1,
        figure=fig,
        left=0.2,
        bottom=0.05,
        right=0.975,
        top=1.1,
        wspace=0,
        hspace=0,
        height_ratios=[6, 1, 2, 1],
    )

    # Plot image roi
    ax0 = fig.add_subplot(gs1[0])
    m = ax0.imshow(data_bin["images"][i].values[roi_s]*(1-mask[roi_s]), vmin=allmin, vmax=allmax)
    plt.colorbar(m, ax=ax0, pad=0.045, location="bottom")

    # Plot 1d azimuthal integration
    ax1 = fig.add_subplot(gs1[1])
    tmp = data_bin.i1d[i]
    ax1.plot(data_bin.q, tmp)
    ax1.set_xlabel("q")
    ax1.set_ylabel("Mean Intensity")
    ax1.set_xlim([q0, q1])
    ax1.set_ylim([allmin, allmax])
    ax1.set_yscale("log")
    ax1.grid()
    
    ax2 = fig.add_subplot(gs1[2])
    vmin, vmax = np.nanpercentile(data_bin["i1dlog"], [.1, 99.9])
    data_bin["i1dlog"].plot.contourf(
        x=scan_axis,
        y="q",
        ax=ax2,
        cmap="viridis",
        add_colorbar=False,
        vmin=vmin,
        vmax=vmax,
        levels=200,
        ylim = [q0,q1]
    )
    ax2.vlines(data_bin[scan_axis].values[i], q0, q1,'r')
    ax2.hlines(q0, data_bin[scan_axis].min(),data_bin[scan_axis].max(),'w',linestyles='dashed')
    ax2.hlines(q1, data_bin[scan_axis].min(),data_bin[scan_axis].max(),'w',linestyles='dashed')

    # Plot SAXS Intensity
    ax3 = fig.add_subplot(gs1[3])
    ax3.plot(data_bin[scan_axis].values, data_bin["saxs"].values)
    ax3.scatter(data_bin[scan_axis].values[i], data_bin["saxs"].values[i], 20, color="r")
    ax3.set_xlabel(scan_axis)
    ax3.set_ylabel("Mean intensity")
    ax3.grid()
    ax3.set_xlim(data_bin[scan_axis].min(),data_bin[scan_axis].max())

    # Title and fname
    ax0.set_title(f"%s:  %s = %s" % (scan, scan_axis, data_bin[scan_axis].values[i]))

    # Save
    fname = join(folder_gif, "Hysteresis_%s_%s_%03d_%s.png" % (scan, chi_mode, i, USER))
    im_fnames.append(fname)
    plt.savefig(fname)
    plt.close()


# Create gif for 1d AI
fname = f"SAXS_%s_%s_%s.gif" % (scan, chi_mode, USER)
gif_path = join(fsave, fname)
print("Saving gif:%s" % gif_path)
helper.create_gif(im_fnames,gif_path,fps=2)
print("Done!")

In [ ]:
# Drop images
data_bin2 = data_bin.drop_vars(["images"])

# Save log
folder = join(fsave, "Logs")
helper.create_folder(folder)
fname = join(folder, "Log_Hysteresis_Scan_%03d_%s_%s.nc" % (scan_id, chi_mode, USER))

print(f"Saving:", fname)
data_bin2.to_netcdf(fname)

# Batch processing

## Loading and pre-processing

In [ ]:
def worker(
    samplefolder,
    scan,
    mask,
    ai,
    scan_axis,
    binning=None,
    keys=None,
    sort=False,
):
    # Load scan data
    datafolder = join(BASEFOLDER, samplefolder)
    exp = preprocess_exp(datafolder, "_OF", keys=keys, sort=sort)

    # Load background images
    exp_bg = preprocess_exp(datafolder, "_BG", keys=keys)

    # Normalize images
    dark = np.mean(np.stack(exp_bg["images"]), axis=0)
    images = np.stack(exp.images) - dark
    images = images / np.broadcast_to(np.array(exp["diode_sum_mean"]), images.T.shape).T

    # Create xarray dataset
    data = xr.Dataset()
    data["images"] = xr.DataArray(images, dims=["index", "y", "x"])
    data[scan_axis] = xr.DataArray(exp[scan_axis], dims=["index"])

    # Do 2d Azimuthal integration of all images and append them to list
    list_i2d = []
    for im in tqdm(data["images"].values):
        i2d, q, chi = ai.integrate2d(im, 500, 90, dummy=np.nan, mask=mask,method="bbox")
        list_i2d.append(i2d)

    # Add to xarray
    data["q"] = q
    data["chi"] = chi
    data["i2d"] = xr.DataArray(list_i2d, dims=["index", "chi", "q"])
    data["i1d"] = data.i2d.where(sel_chi, drop=True).mean("chi")

    # Averaging of same scan axis values or binning
    if binning is None:
        _, count = np.unique(data[scan_axis].values, return_counts=True)
        if np.any(count > 1):
            data_bin = data.groupby(scan_axis).mean()
        else:
            data_bin = data.swap_dims({"index": scan_axis})

    else:
        # Execute binning
        data_bin = data.groupby_bins(scan_axis, bins).mean()

        # Rename binned values, drop intervals as those cannot be save in h5
        bin_scan_axis = scan_axis + "_bins"
        data_bin = data_bin.swap_dims({bin_scan_axis: scan_axis})
        data_bin = data_bin.drop(bin_scan_axis)

    # Add log plot
    data_bin["i1dlog"] = np.log10(data_bin["i1d"]+ 1)
    
    # Add AI mask
    data_bin["mask"] = xr.DataArray(mask, dims=["y", "x"])

    # Add file scan labels
    data_bin["scan"] = scan

    # Direction of "time"
    if np.sum(data[scan_axis][1:].values - data[scan_axis][0:-1].values) >= 0:
        data_bin["order"] = 1
    elif np.sum(data[scan_axis][1:].values - data[scan_axis][0:-1].values) < 0:
        data_bin["order"] = -1


    # Drop images to save disk space
    data_save = data_bin.drop_vars(["images"])

    # Save log
    folder = join(fsave, "Logs")
    fname = join(folder, "Log_Hysteresis_Scan_%s_%s_%s.nc" % (scan, chi_mode, USER))
    print(f"Saving:", fname)
    data_save.to_netcdf(fname)

    return data_bin

In [ ]:
# Name of scans (see pad)
scans = [
    "I7_006",
    "I7_007",
    "I7_008",
    "I7_010",
]

# Analysis options
bins = []

# Setup xarray for scans
data_scans = []

# Loop over scans
for scan in tqdm(scans):
    # Process data
    samplefolder = join(sample, scan)
    data = worker(
        samplefolder,
        scan,
        mask,
        ai,
        scan_axis,
        binning=None,
        keys=extra_keys,
        sort=False,
    )

    # Add to xarray list
    data_scans.append(data)

# Combine separate xarrays
data_scans = xr.concat(data_scans, dim="scanid")
data_scans

## Calc and plot SAXS

In [ ]:
# Do you want to norm the hysteresis?
normalization = True

# Select relevant q-range for averaging
# q0, q1 = 0.003, 0.04

# Get SAXS from q-range
sel = (data_scans.q > q0) * (data_scans.q < q1)
data_scans["saxs"] = data_scans.i1d.where(sel, drop=True).mean("q")

# Plot all SAXS images individually
for scanid in data_scans["scanid"].values:
    fig, ax = plt.subplots()

    # Normalize?
    if normalization is True:
        tmp_data = data_scans["saxs"][scanid].values
        tmp_data = norm(data_scans["saxs"][scanid].values[1:])
    else:
        tmp_data = data_scans["saxs"][scanid].values[1:]

    ax.plot(data_scans[scan_axis].values[1:], tmp_data, "o-")
    ax.set_xlabel(scan_axis)
    ax.set_ylabel("Integrated SAXS")
    ax.set_title("Scan: %s" % data_scans["scan"][scanid].values)
    ax.grid()

    # Save fig
    fname = join(
        fsave,
        "Hysteresis_%s_%s_%s.png" % (data_scans["scan"][scanid].values, chi_mode, USER),
    )
    print("Saving: %s" % fname)
    plt.savefig(fname)
    plt.close()

# Plot them together
fig, ax = plt.subplots()
for scanid in data_scans["scanid"].values:
    # Normalize?
    if normalization is True:
        tmp_data = data_scans["saxs"][scanid].values
        tmp_data = norm(tmp_data[1:])
    else:
        tmp_data = data_scans["saxs"][scanid].values[1:]
    ax.plot(
        data_scans[scan_axis].values[1:],
        tmp_data,
        "o-",
        label=data_scans["scan"][scanid].values,
    )

ax.set_xlabel(scan_axis)
ax.set_ylabel("Integrated SAXS")
ax.grid()
ax.legend()

# Save fig
fname = join(
    fsave,
    "Hysteresis_%s_%s_%s_%s.png"
    % (data_scans["scan"][0].values, data_scans["scan"][-1].values, chi_mode, USER),
)
plt.savefig(fname)

In [ ]:
# Export gif for all scans individually
for scan_id in tqdm(data_scans["scanid"].values):
    tmp_xr = data_scans.sel(scanid = scan_id)
    
    if tmp_xr["order"] == 1:
        tmp_xr = tmp_xr.reindex(magnet_mean=data_scans.magnet_mean[::-1])

    # Find max and min considering all images
    allmin, allmax = np.nanpercentile(tmp_xr["i2d"].values, [1, 99])
    if allmin < .1:
        allmin = .1
    print("Min: %d Max: %d" % (allmin, allmax))

    # Create folder for gif single frames
    folder_gif = helper.create_folder(join(fsave, "Scan_%s" % tmp_xr["scan"].values))

    # Normalize?
    if normalization is True:
        tmp_data = tmp_xr["saxs"].values
        tmp_data = norm(tmp_data[1:])
    else:
        tmp_data = tmp_xr["saxs"].values
        tmp_data = tmp_data[1:]

    im_fnames = []
    for i in tqdm(range(1, len(tmp_xr[scan_axis].values))):
        # Plot for averaged image
        fig = plt.figure(figsize=(6, 10))
        gs1 = gridspec.GridSpec(
            4,
            1,
            figure=fig,
            left=0.2,
            bottom=0.05,
            right=0.975,
            top=1.1,
            wspace=0,
            hspace=0,
            height_ratios=[6, 1, 2, 1],
        )
    
        # Plot image roi
        ax0 = fig.add_subplot(gs1[0])
        m = ax0.imshow(tmp_xr["images"][i].values[roi_s]*(1-mask[roi_s]), vmin=allmin, vmax=allmax)
        plt.colorbar(m, ax=ax0, pad=0.045, location="bottom")
    
        # Plot 1d azimuthal integration
        ax1 = fig.add_subplot(gs1[1])
        tmp = tmp_xr.i1d[i]
        ax1.plot(tmp_xr.q, tmp)
        ax1.set_xlabel("q")
        ax1.set_ylabel("Mean Intensity")
        ax1.set_xlim([q0, q1])
        ax1.set_ylim([allmin, allmax])
        ax1.set_yscale("log")
        ax1.grid()
        
        ax2 = fig.add_subplot(gs1[2])
        vmin, vmax = np.nanpercentile(tmp_xr["i1dlog"], [1, 99])
        tmp_xr["i1dlog"].plot.contourf(
            x=scan_axis,
            y="q",
            ax=ax2,
            cmap="viridis",
            add_colorbar=False,
            vmin=vmin,
            vmax=vmax,
            levels=300,
            ylim = [q0,q1]
        )
        ax2.vlines(tmp_xr[scan_axis].values[i], q0, q1,'r')
        ax2.hlines(q0, tmp_xr[scan_axis].min(),data_bin[scan_axis].max(),'w',linestyles='dashed')
        ax2.hlines(q1, tmp_xr[scan_axis].min(),data_bin[scan_axis].max(),'w',linestyles='dashed')
    
        # Plot SAXS Intensity
        ax3 = fig.add_subplot(gs1[3])
        ax3.plot(tmp_xr[scan_axis].values, tmp_xr["saxs"].values)
        ax3.scatter(tmp_xr[scan_axis].values[i], tmp_xr["saxs"].values[i], 20, color="r")
        ax3.set_xlabel(scan_axis)
        ax3.set_ylabel("Mean intensity")
        ax3.grid()
        ax3.set_xlim(tmp_xr[scan_axis].min(),data_bin[scan_axis].max())
    
        # Title and fname
        ax0.set_title(f"%s:  %s = %s" % (tmp_xr["scan"].values, scan_axis, tmp_xr[scan_axis].values[i]))

        # Save
        fname = join(
            folder_gif,
            "Hysteresis_%s_%03d_%s_%s.png"
            % (tmp_xr["scan"].values, i, chi_mode, USER),
        )
        im_fnames.append(fname)
        plt.savefig(fname)
        plt.close()

    # Create gif for 1d AI
    var = [imageio.imread(file) for file in im_fnames]
    fname = f"Hysteresis_%s_%s_%s.gif" % (
        tmp_xr["scan"].values,
        chi_mode,
        USER,
    )
    gif_path = join(fsave, fname)
    print("Saving gif:%s" % gif_path)
    helper.create_gif(im_fnames,gif_path,fps=2)
    print("Done!")

In [ ]:
plt.close("all")